Celda 1: Instalación de dependencias

# Pre-entrega 5: Agente de razonamiento cíclico con memoria persistente
Agente ReAct con LangGraph: decide solo qué herramientas usar, reintenta ante errores y recuerda la conversación por thread_id gracias a SqliteSaver.

## 1. Instalación de librerías


In [5]:
!pip install -q langgraph langgraph-checkpoint-sqlite aiosqlite langchain-core langchain-google-genai

Celda 2: Configuración de la API key

In [35]:
import os
import sys
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

GEMINI_MODEL = "gemini-3-flash-preview"
DB_CHECKPOINTS = "checkpoints.db"   # archivo SQLite donde se guarda la memoria
RECURSION_LIMIT = 10                # techo de pasos por consulta (evita bucles infinitos)
MAX_MENSAJES = 20                   # máximo de mensajes que se le pasan al modelo

print("Python", sys.version.split()[0])

Python 3.13.18


## 3. Configuración de logs

In [36]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
for ruidoso in ("httpx", "google_genai", "aiosqlite"):
    logging.getLogger(ruidoso).setLevel(logging.WARNING)
logger = logging.getLogger("agente")

## 4. Base de datos simulada
Clientes y pedidos de una tienda. Hay dos clientes llamados Martín, para probar ambigüedades.

In [37]:
from typing import TypedDict


class Cliente(TypedDict):
    cliente_id: int
    nombre: str
    ciudad: str


class Pedido(TypedDict):
    pedido_id: int
    cliente_id: int
    fecha: str
    total: int
    estado: str
    productos: list[str]


CLIENTES: list[Cliente] = [
    {"cliente_id": 101, "nombre": "Lucía Fernández", "ciudad": "Rosario"},
    {"cliente_id": 102, "nombre": "Martín Gómez", "ciudad": "Córdoba"},
    {"cliente_id": 103, "nombre": "Martín Sosa", "ciudad": "Mendoza"},
    {"cliente_id": 104, "nombre": "Carla Ruiz", "ciudad": "La Plata"},
]

PEDIDOS: list[Pedido] = [
    {"pedido_id": 5001, "cliente_id": 102, "fecha": "2026-08-14", "total": 4500,
     "estado": "entregado", "productos": ["Mouse inalámbrico", "Pad gamer"]},
    {"pedido_id": 5007, "cliente_id": 102, "fecha": "2026-09-30", "total": 6000,
     "estado": "entregado", "productos": ["Teclado mecánico"]},
    {"pedido_id": 5012, "cliente_id": 102, "fecha": "2026-10-21", "total": 4000,
     "estado": "en camino", "productos": ["Auriculares USB"]},
    {"pedido_id": 5003, "cliente_id": 101, "fecha": "2026-09-02", "total": 12800,
     "estado": "entregado", "productos": ["Disco SSD 1TB"]},
    {"pedido_id": 5010, "cliente_id": 101, "fecha": "2026-10-11", "total": 3200,
     "estado": "cancelado", "productos": ["Cable HDMI", "Adaptador USB-C"]},
    {"pedido_id": 5009, "cliente_id": 103, "fecha": "2026-10-05", "total": 21000,
     "estado": "entregado", "productos": ["Monitor 24 pulgadas"]},
]

# Simula una falla pasajera: la primera consulta de pedidos del cliente 101 da timeout
FALLAS_SIMULADAS: set[int] = {101}

## 5. Herramientas (@tool)
El modelo elige qué herramienta usar leyendo solo su docstring, por eso son tan detallados.

In [38]:
import asyncio
from typing import Any

from langchain_core.tools import tool


@tool
async def buscar_cliente(nombre: str) -> dict[str, Any]:
    """Busca clientes por nombre o apellido (no hace falta el nombre completo).

    Usala cuando el usuario mencione a un cliente por su nombre y necesites su cliente_id
    para consultar pedidos. Devuelve la lista de coincidencias con cliente_id, nombre y ciudad.
    Si hay varias coincidencias, NO elijas una: preguntale al usuario cuál es.
    Si no hay coincidencias, devuelve un error.
    """
    await asyncio.sleep(0.2)  # latencia simulada de la base de datos
    texto = nombre.strip().lower()
    coincidencias = [c for c in CLIENTES if texto in c["nombre"].lower()]
    logger.info("🔧 buscar_cliente(%r) -> %d coincidencia(s)", nombre, len(coincidencias))

    if not coincidencias:
        return {"error": f"No existe ningún cliente que coincida con '{nombre}'."}
    if len(coincidencias) > 1:
        return {"coincidencias": coincidencias,
                "aviso": "Hay más de un cliente con ese nombre. Pedile al usuario que aclare cuál."}
    return {"coincidencias": coincidencias}


@tool
async def buscar_pedidos(cliente_id: int) -> dict[str, Any]:
    """Devuelve el resumen de pedidos de un cliente a partir de su cliente_id numérico.

    Usala para responder cuántos pedidos tiene un cliente, cuánto gastó en total o cuál
    fue su último pedido. Devuelve la cantidad de pedidos, el total en pesos y la lista
    de pedidos (pedido_id, fecha, total y estado) ordenada del más viejo al más nuevo.
    Si devuelve un error temporal, volvé a llamarla con el mismo cliente_id.
    """
    await asyncio.sleep(0.2)
    if cliente_id in FALLAS_SIMULADAS:
        FALLAS_SIMULADAS.discard(cliente_id)
        logger.warning("🔧 buscar_pedidos(%d) -> timeout simulado", cliente_id)
        return {"error": "Timeout temporal de la base de datos. Reintentá la misma consulta."}

    if not any(c["cliente_id"] == cliente_id for c in CLIENTES):
        logger.info("🔧 buscar_pedidos(%d) -> cliente inexistente", cliente_id)
        return {"error": f"No existe el cliente_id {cliente_id}."}

    pedidos = sorted((p for p in PEDIDOS if p["cliente_id"] == cliente_id), key=lambda p: p["fecha"])
    logger.info("🔧 buscar_pedidos(%d) -> %d pedido(s)", cliente_id, len(pedidos))
    return {
        "cliente_id": cliente_id,
        "pedidos": len(pedidos),
        "total": sum(p["total"] for p in pedidos),
        "lista": [{k: p[k] for k in ("pedido_id", "fecha", "total", "estado")} for p in pedidos],
    }


@tool
async def detalle_pedido(pedido_id: int) -> dict[str, Any]:
    """Devuelve el detalle completo de un pedido a partir de su pedido_id numérico:
    fecha, total, estado de envío y productos comprados.

    Usala cuando el usuario pregunte qué compró en un pedido puntual o en qué estado está.
    El pedido_id se obtiene antes con buscar_pedidos.
    """
    await asyncio.sleep(0.2)
    pedido = next((p for p in PEDIDOS if p["pedido_id"] == pedido_id), None)
    logger.info(" detalle_pedido(%d) -> %s", pedido_id, "encontrado" if pedido else "no existe")
    if pedido is None:
        return {"error": f"No existe el pedido {pedido_id}."}
    return dict(pedido)


HERRAMIENTAS = [buscar_cliente, buscar_pedidos, detalle_pedido]

## 6. Prueba directa de las herramientas (sin LLM)

In [39]:
print(await buscar_cliente.ainvoke({"nombre": "Martín"}))
print(await buscar_cliente.ainvoke({"nombre": "Gómez"}))
print(await buscar_pedidos.ainvoke({"cliente_id": 102}))
print(await detalle_pedido.ainvoke({"pedido_id": 9999}))

23:26:33 | INFO    | agente | 🔧 buscar_cliente('Martín') -> 2 coincidencia(s)
23:26:33 | INFO    | agente | 🔧 buscar_cliente('Gómez') -> 1 coincidencia(s)


{'coincidencias': [{'cliente_id': 102, 'nombre': 'Martín Gómez', 'ciudad': 'Córdoba'}, {'cliente_id': 103, 'nombre': 'Martín Sosa', 'ciudad': 'Mendoza'}], 'aviso': 'Hay más de un cliente con ese nombre. Pedile al usuario que aclare cuál.'}


23:26:33 | INFO    | agente | 🔧 buscar_pedidos(102) -> 3 pedido(s)


{'coincidencias': [{'cliente_id': 102, 'nombre': 'Martín Gómez', 'ciudad': 'Córdoba'}]}


23:26:33 | INFO    | agente |  detalle_pedido(9999) -> no existe


{'cliente_id': 102, 'pedidos': 3, 'total': 14500, 'lista': [{'pedido_id': 5001, 'fecha': '2026-08-14', 'total': 4500, 'estado': 'entregado'}, {'pedido_id': 5007, 'fecha': '2026-09-30', 'total': 6000, 'estado': 'entregado'}, {'pedido_id': 5012, 'fecha': '2026-10-21', 'total': 4000, 'estado': 'en camino'}]}
{'error': 'No existe el pedido 9999.'}


## 7. Estado del agente
Hereda de MessagesState, que acumula los mensajes con el reducer add_messages. Se agrega un contador de llamadas al LLM que se acumula con operator.add.

In [40]:
import operator
from typing import Annotated

from langgraph.graph import MessagesState


class EstadoAgente(MessagesState):
    """Estado del grafo: la lista de mensajes (heredada) más un contador de pasos del modelo."""
    llamadas_llm: Annotated[int, operator.add]

## 8. Modelo con herramientas (bind_tools) y prompt de sistema

In [41]:
from langchain_google_genai import ChatGoogleGenerativeAI

SYSTEM_PROMPT = """Sos el asistente de atención al cliente de una tienda de tecnología.
Respondés consultas sobre clientes y pedidos usando SOLO las herramientas disponibles.

Reglas:
- Nunca inventes datos: si no los tenés, usá una herramienta.
- Si el usuario nombra a un cliente, primero buscá su cliente_id con buscar_cliente.
- Si una herramienta devuelve un error temporal, volvé a intentarlo una vez.
- Si hay varios clientes que coinciden, no elijas vos: preguntale al usuario cuál es.
- Los montos son en pesos argentinos; escribilos con separador de miles, por ejemplo $14.500.
- Respondé en español, de forma breve y clara."""

llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0, max_retries=5)
llm_con_herramientas = llm.bind_tools(HERRAMIENTAS)
logger.info("LLM %s vinculado a %d herramientas: %s",
            GEMINI_MODEL, len(HERRAMIENTAS), [h.name for h in HERRAMIENTAS])

23:26:36 | INFO    | agente | LLM gemini-3-flash-preview vinculado a 3 herramientas: ['buscar_cliente', 'buscar_pedidos', 'detalle_pedido']


## 9. Nodo del modelo
Recorta el historial a los últimos mensajes antes de llamar al LLM, para que el estado no crezca sin límite.

In [13]:
from langchain_core.messages import AIMessage, BaseMessage, SystemMessage, trim_messages


def texto_de(mensaje: BaseMessage) -> str:
    """Extrae el texto de un mensaje (Gemini a veces devuelve el contenido en bloques)."""
    contenido = mensaje.content
    if isinstance(contenido, str):
        return contenido
    return "".join(b.get("text", "") for b in contenido if isinstance(b, dict) and b.get("type") == "text")


async def nodo_agente(state: EstadoAgente) -> dict[str, Any]:
    """Llama al LLM; este decide si responder o pedir una herramienta."""
    historial = trim_messages(
        state["messages"],
        strategy="last",
        token_counter=len,        # cuenta mensajes
        max_tokens=MAX_MENSAJES,
        start_on="human",         # el recorte siempre arranca en un mensaje del usuario
    )
    respuesta: AIMessage = await llm_con_herramientas.ainvoke([SystemMessage(SYSTEM_PROMPT), *historial])

    if respuesta.tool_calls:
        for llamada in respuesta.tool_calls:
            logger.info("🤖 El agente decide usar: %s(%s)", llamada["name"], llamada["args"])
    else:
        logger.info("🤖 El agente responde sin herramientas")

    return {"messages": [respuesta], "llamadas_llm": 1}

## 10. Grafo: agente <-> herramientas con arista condicional

In [14]:
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

builder = StateGraph(EstadoAgente)
builder.add_node("agente", nodo_agente)
builder.add_node("tools", ToolNode(HERRAMIENTAS, handle_tool_errors=True))

builder.add_edge(START, "agente")
builder.add_conditional_edges("agente", tools_condition)  # si pidió herramienta -> tools, si no -> END
builder.add_edge("tools", "agente")                       # el resultado vuelve al agente (ciclo)

## 11. Persistencia con SqliteSaver
La memoria de cada conversación se guarda en checkpoints.db, identificada por su thread_id.

In [15]:
import aiosqlite
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

conexion = await aiosqlite.connect(DB_CHECKPOINTS)
checkpointer = AsyncSqliteSaver(conexion)

grafo = builder.compile(checkpointer=checkpointer)
logger.info("Grafo compilado con persistencia en %s", DB_CHECKPOINTS)

print(grafo.get_graph().draw_mermaid())

23:16:28 | INFO    | agente | Grafo compilado con persistencia en checkpoints.db


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agente(agente)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agente;
	agente -.-> __end__;
	agente -.-> tools;
	tools --> agente;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 12. Función para consultar al agente

In [16]:
from langchain_core.messages import HumanMessage
from langgraph.errors import GraphRecursionError


async def preguntar(pregunta: str, thread_id: str, intentos: int = 3, espera: int = 30) -> str | None:
    """Envía una pregunta al agente dentro de una conversación (thread_id)."""
    config = {"configurable": {"thread_id": thread_id}, "recursion_limit": RECURSION_LIMIT}
    print(f"\n [{thread_id}] {pregunta}")

    for intento in range(1, intentos + 1):
        try:
            resultado = await grafo.ainvoke({"messages": [HumanMessage(pregunta)]}, config)
            respuesta = texto_de(resultado["messages"][-1])
            print(f"{respuesta}")
            return respuesta
        except GraphRecursionError:
            print(f"Se alcanzó el límite de {RECURSION_LIMIT} pasos sin respuesta final")
            return None
        except Exception as e:
            if "503" in str(e) and intento < intentos:
                logger.warning("Gemini saturado (503). Reintento %d de %d en %ds...",
                               intento + 1, intentos, espera)
                await asyncio.sleep(espera)
            else:
                print(f"Error controlado ({type(e).__name__}): {e}")
                return None


await preguntar("Hola, ¿en qué me podés ayudar?", thread_id="prueba-inicial")


 [prueba-inicial] Hola, ¿en qué me podés ayudar?


23:16:33 | INFO    | agente | 🤖 El agente responde sin herramientas


¡Hola! Puedo ayudarte con las siguientes consultas sobre nuestra tienda de tecnología:

* **Buscar clientes** por nombre o apellido.
* **Consultar pedidos de un cliente**: ver la cantidad de pedidos realizados, el total gastado o su último pedido.
* **Ver detalles de un pedido**: conocer los productos comprados y el estado del envío.

¿En qué te puedo colaborar hoy?


'¡Hola! Puedo ayudarte con las siguientes consultas sobre nuestra tienda de tecnología:\n\n* **Buscar clientes** por nombre o apellido.\n* **Consultar pedidos de un cliente**: ver la cantidad de pedidos realizados, el total gastado o su último pedido.\n* **Ver detalles de un pedido**: conocer los productos comprados y el estado del envío.\n\n¿En qué te puedo colaborar hoy?'

## Limpieza: las pruebas arrancan desde cero
Borra las conversaciones anteriores del checkpointer y reactiva la falla simulada, para que la traza quede limpia si se vuelve a correr.

In [42]:
for thread in ["sesion-1", "sesion-2", "sesion-3", "sesion-nueva"]:
    await checkpointer.adelete_thread(thread)
FALLAS_SIMULADAS.clear()
FALLAS_SIMULADAS.add(101)
logger.info("Conversaciones de prueba borradas")

23:26:52 | INFO    | agente | Conversaciones de prueba borradas


## 13. Prueba 1: razonamiento multi-paso
El usuario da un nombre, no un ID: el agente tiene que buscar el cliente y después sus pedidos (2 herramientas).

In [43]:
await preguntar("¿Cuántos pedidos tuvo Martín Gómez y cuál fue el total?", thread_id="sesion-1")


 [sesion-1] ¿Cuántos pedidos tuvo Martín Gómez y cuál fue el total?


23:26:56 | INFO    | agente | 🤖 El agente decide usar: buscar_cliente({'nombre': 'Martín Gómez'})
23:26:56 | INFO    | agente | 🔧 buscar_cliente('Martín Gómez') -> 1 coincidencia(s)
23:26:57 | INFO    | agente | 🤖 El agente decide usar: buscar_pedidos({'cliente_id': 102})
23:26:58 | INFO    | agente | 🔧 buscar_pedidos(102) -> 3 pedido(s)
23:26:58 | INFO    | agente | 🤖 El agente responde sin herramientas


Martín Gómez tuvo 3 pedidos con un total de $14.500.


'Martín Gómez tuvo 3 pedidos con un total de $14.500.'

## 14. Prueba 2: memoria con el mismo thread_id
La pregunta no menciona al cliente: el agente tiene que recordar de quién se hablaba.

In [44]:
await preguntar("¿Y el último? ¿Qué compró y en qué estado está?", thread_id="sesion-1")


 [sesion-1] ¿Y el último? ¿Qué compró y en qué estado está?


23:27:03 | INFO    | agente | 🤖 El agente decide usar: detalle_pedido({'pedido_id': 5012})
23:27:03 | INFO    | agente |  detalle_pedido(5012) -> encontrado
23:27:04 | INFO    | agente | 🤖 El agente responde sin herramientas


Su último pedido (N° 5012) fue por unos Auriculares USB, por un total de $4.000, y se encuentra en camino.


'Su último pedido (N° 5012) fue por unos Auriculares USB, por un total de $4.000, y se encuentra en camino.'

## 15. Prueba 3: ciclo de retorno ante un error
La primera consulta de pedidos de Lucía falla con un timeout simulado; el agente debe reintentar.

In [45]:
await preguntar("¿Cuánto gastó en total Lucía Fernández?", thread_id="sesion-2")


 [sesion-2] ¿Cuánto gastó en total Lucía Fernández?


23:27:07 | INFO    | agente | 🤖 El agente decide usar: buscar_cliente({'nombre': 'Lucía Fernández'})
23:27:07 | INFO    | agente | 🔧 buscar_cliente('Lucía Fernández') -> 1 coincidencia(s)


Error controlado (GoogleRateLimitError): Error calling model 'gemini-3-flash-preview' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3-flash, description: Generate content requests per minute limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 34.268276951s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations'

## 16. Prueba 4: información ambigua -> pide aclaración

In [46]:
await preguntar("¿Cuántos pedidos tiene Martín?", thread_id="sesion-3")
await preguntar("El de Mendoza", thread_id="sesion-3")


 [sesion-3] ¿Cuántos pedidos tiene Martín?
Error controlado (GoogleRateLimitError): Error calling model 'gemini-3-flash-preview' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3-flash, description: Generate content requests per minute limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 8.494471314s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis

23:27:56 | INFO    | agente | 🤖 El agente decide usar: buscar_cliente({'nombre': 'Martín'})
23:27:56 | INFO    | agente | 🔧 buscar_cliente('Martín') -> 2 coincidencia(s)
23:27:57 | INFO    | agente | 🤖 El agente decide usar: buscar_pedidos({'cliente_id': 103})
23:27:58 | INFO    | agente | 🔧 buscar_pedidos(103) -> 1 pedido(s)
23:27:59 | INFO    | agente | 🤖 El agente responde sin herramientas


Martín Sosa, de Mendoza, tiene 1 pedido realizado por un total de $21.000.


'Martín Sosa, de Mendoza, tiene 1 pedido realizado por un total de $21.000.'

In [49]:

for thread in ["sesion-2", "sesion-3"]:
    await checkpointer.adelete_thread(thread)
FALLAS_SIMULADAS.clear()
FALLAS_SIMULADAS.add(101)

await preguntar("¿Cuánto gastó en total Lucía Fernández?", thread_id="sesion-2")
await asyncio.sleep(60)
await preguntar("¿Cuántos pedidos tiene Martín?", thread_id="sesion-3")
await asyncio.sleep(60)
await preguntar("El de Mendoza", thread_id="sesion-3")


 [sesion-2] ¿Cuánto gastó en total Lucía Fernández?


23:29:25 | INFO    | agente | 🤖 El agente decide usar: buscar_cliente({'nombre': 'Lucía Fernández'})
23:29:25 | INFO    | agente | 🔧 buscar_cliente('Lucía Fernández') -> 1 coincidencia(s)
23:29:26 | INFO    | agente | 🤖 El agente decide usar: buscar_pedidos({'cliente_id': 101})
23:29:26 | WARNING | agente | 🔧 buscar_pedidos(101) -> timeout simulado
23:29:27 | INFO    | agente | 🤖 El agente decide usar: buscar_pedidos({'cliente_id': 101})
23:29:27 | INFO    | agente | 🔧 buscar_pedidos(101) -> 2 pedido(s)
23:29:28 | INFO    | agente | 🤖 El agente responde sin herramientas


Lucía Fernández gastó un total de $16.000 en sus 2 pedidos realizados.

 [sesion-3] ¿Cuántos pedidos tiene Martín?


23:30:29 | INFO    | agente | 🤖 El agente decide usar: buscar_cliente({'nombre': 'Martín'})
23:30:30 | INFO    | agente | 🔧 buscar_cliente('Martín') -> 2 coincidencia(s)
23:30:31 | INFO    | agente | 🤖 El agente responde sin herramientas


Hay dos clientes llamados Martín. ¿A cuál te referís?

1. **Martín Gómez** (Córdoba)
2. **Martín Sosa** (Mendoza)

 [sesion-3] El de Mendoza


23:31:32 | INFO    | agente | 🤖 El agente decide usar: buscar_pedidos({'cliente_id': 103})
23:31:32 | INFO    | agente | 🔧 buscar_pedidos(103) -> 1 pedido(s)
23:31:33 | INFO    | agente | 🤖 El agente responde sin herramientas


Martín Sosa tiene 1 pedido por un total de $21.000.


'Martín Sosa tiene 1 pedido por un total de $21.000.'

## 17. Prueba 5: thread_id nuevo, sin memoria
La misma pregunta de la prueba 2, pero en otra conversación: el agente no debería saber de quién se habla.

In [47]:
await preguntar("¿Y el último? ¿Qué compró y en qué estado está?", thread_id="sesion-nueva")


 [sesion-nueva] ¿Y el último? ¿Qué compró y en qué estado está?


23:28:08 | INFO    | agente | 🤖 El agente responde sin herramientas


Para poder ayudarte, ¿podrías decirme el nombre del cliente? Así podré buscar su último pedido y decirte qué compró y en qué estado se encuentra.


'Para poder ayudarte, ¿podrías decirme el nombre del cliente? Así podré buscar su último pedido y decirte qué compró y en qué estado se encuentra.'

## 18. Traza ReAct: lectura del estado guardado y exportación a JSON

In [50]:
import ast
import json
from langchain_core.messages import ToolMessage


def parsear(contenido: str) -> Any:
    """Convierte la salida de una herramienta a dict si se puede."""
    for cargar in (json.loads, ast.literal_eval):
        try:
            return cargar(contenido)
        except Exception:
            pass
    return contenido


def mensaje_a_paso(m: BaseMessage) -> dict[str, Any]:
    if isinstance(m, HumanMessage):
        return {"tipo": "usuario", "contenido": texto_de(m)}
    if isinstance(m, ToolMessage):
        return {"tipo": "resultado_herramienta", "herramienta": m.name, "contenido": parsear(m.content)}
    if isinstance(m, AIMessage) and m.tool_calls:
        return {"tipo": "agente_decide_herramienta",
                "llamadas": [{"herramienta": t["name"], "argumentos": t["args"]} for t in m.tool_calls]}
    return {"tipo": "respuesta_final", "contenido": texto_de(m)}


async def obtener_traza(thread_id: str) -> dict[str, Any]:
    """Lee del checkpointer todo lo que pasó en una conversación."""
    estado = await grafo.aget_state({"configurable": {"thread_id": thread_id}})
    pasos = [mensaje_a_paso(m) for m in estado.values.get("messages", [])]
    llamadas = sum(len(p["llamadas"]) for p in pasos if p["tipo"] == "agente_decide_herramienta")
    return {"thread_id": thread_id,
            "llamadas_llm": estado.values.get("llamadas_llm", 0),
            "llamadas_herramientas": llamadas,
            "pasos": pasos}


def imprimir_traza(traza: dict[str, Any]) -> None:
    print(f"\n===== {traza['thread_id']} | LLM: {traza['llamadas_llm']} | "
          f"herramientas: {traza['llamadas_herramientas']} =====")
    for p in traza["pasos"]:
        if p["tipo"] == "usuario":
            print(f'Usuario: "{p["contenido"]}"')
        elif p["tipo"] == "agente_decide_herramienta":
            for ll in p["llamadas"]:
                args = ", ".join(f"{k}={v!r}" for k, v in ll["argumentos"].items())
                print(f"  → El agente decide usar: {ll['herramienta']}({args})")
        elif p["tipo"] == "resultado_herramienta":
            print(f"  → La herramienta devuelve: {json.dumps(p['contenido'], ensure_ascii=False)[:150]}")
        else:
            print(f'Respuesta: "{p["contenido"]}"')


THREADS = ["sesion-1", "sesion-2", "sesion-3", "sesion-nueva"]
trazas = [await obtener_traza(t) for t in THREADS]

for traza in trazas:
    imprimir_traza(traza)

with open("traza_react.json", "w", encoding="utf-8") as f:
    json.dump({"modelo": GEMINI_MODEL, "recursion_limit": RECURSION_LIMIT, "conversaciones": trazas},
              f, ensure_ascii=False, indent=2)
print("\n✅ Traza guardada en traza_react.json")


===== sesion-1 | LLM: 5 | herramientas: 3 =====
Usuario: "¿Cuántos pedidos tuvo Martín Gómez y cuál fue el total?"
  → El agente decide usar: buscar_cliente(nombre='Martín Gómez')
  → La herramienta devuelve: {"coincidencias": [{"cliente_id": 102, "nombre": "Martín Gómez", "ciudad": "Córdoba"}]}
  → El agente decide usar: buscar_pedidos(cliente_id=102)
  → La herramienta devuelve: {"cliente_id": 102, "pedidos": 3, "total": 14500, "lista": [{"pedido_id": 5001, "fecha": "2026-08-14", "total": 4500, "estado": "entregado"}, {"pedido
Respuesta: "Martín Gómez tuvo 3 pedidos con un total de $14.500."
Usuario: "¿Y el último? ¿Qué compró y en qué estado está?"
  → El agente decide usar: detalle_pedido(pedido_id=5012)
  → La herramienta devuelve: {"pedido_id": 5012, "cliente_id": 102, "fecha": "2026-10-21", "total": 4000, "estado": "en camino", "productos": ["Auriculares USB"]}
Respuesta: "Su último pedido (N° 5012) fue por unos Auriculares USB, por un total de $4.000, y se encuentra en camino.